# Chapter-Level Classification Experiment

Simplified version of the tree search: given a cause-of-death string,
classify into one of 25 ICD-10 chapters (single letter A–Z).

- **One LLM call per record** (no beam search)
- **Single-CoD records only** (no multi-label)
- Local Qwen2.5-7B-Instruct via LM Studio

In [ ]:
# Cell 1: Imports + Config
from __future__ import annotations

import json
import sys
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from openai import OpenAI

PROJECT_ROOT = Path.cwd().parents[1]
for p in [str(PROJECT_ROOT / "src"), str(PROJECT_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from codllm.config import DataSourceConfig
from codllm.data_handler import BELGIUM_MAPPING, load_source_dataset
from experiments.tree_search.icd10h_tree import build_tree, tree_stats

# --- Paths ---
MASTERLIST_PATH = PROJECT_ROOT / "data" / "ICD10h_Masterlist_2024.xlsx"
CACHE_DIR = PROJECT_ROOT / "data" / "cache" / "chapter_classification"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- Parameters ---
SEED = 42
SAMPLE_FRAC = 0.05
np.random.seed(SEED)

# --- LLM config ---
LLM_BASE_URL = "http://localhost:1234/v1"
LLM_MODEL = "qwen2.5-7b-instruct"  # adjust to match LM Studio
LLM_TEMPERATURE = 0.1

client = OpenAI(base_url=LLM_BASE_URL, api_key="lm-studio")

print(f"Project root: {PROJECT_ROOT}")
print(f"LLM model:    {LLM_MODEL}")

In [ ]:
# Cell 2: Load data — single-CoD records, subsample
belgium_source = DataSourceConfig(
    source_id="belgium_1920_1930",
    path="SOSA_EXTR_1920-1930 (belgium).xlsx",
    mapping_id="belgium",
)

df_all = load_source_dataset(
    source=belgium_source,
    mapping=BELGIUM_MAPPING,
    training_input=["cod"],
    max_labels=6,
    data_raw_dir=str(PROJECT_ROOT / "data" / "raw"),
)

df_all["query"] = df_all["text"].str.replace(r"^cod:\s*", "", regex=True)
df_single = df_all[df_all["y_codes"].apply(len) == 1].copy()
df_single["gold_code"] = df_single["y_codes"].apply(lambda x: x[0])
df_single["gold_chapter"] = df_single["gold_code"].str[0]

df_sample = df_single.sample(frac=SAMPLE_FRAC, random_state=SEED).reset_index(drop=True)

print(f"Total records:       {len(df_all):,}")
print(f"Single-code records: {len(df_single):,} ({len(df_single)/len(df_all)*100:.1f}%)")
print(f"Sample size ({SAMPLE_FRAC*100:.0f}%): {len(df_sample):,}")
print(f"Unique chapters in sample: {df_sample['gold_chapter'].nunique()}")
print()
print(df_sample['gold_chapter'].value_counts().sort_index())

In [ ]:
# Cell 3: Build chapter options from the ICD10h tree
tree = build_tree(MASTERLIST_PATH)

chapter_options = []
for ch in tree.children:
    chapter_options.append((ch.code, ch.label))

options_block = "\n".join(f"  [{code}] {label}" for code, label in chapter_options)

print(f"{len(chapter_options)} chapters:\n")
print(options_block)

In [ ]:
# Cell 4: Classification function — one LLM call per record

SYSTEM_PROMPT = f"""You are a medical coding expert. Given a historical cause-of-death string, classify it into the single most likely ICD-10 chapter.

The chapters are:
{options_block}

The cause-of-death string may be in Dutch, French, Latin, or other historical languages.

Respond with ONLY the single chapter letter (e.g. "A" or "I"). Nothing else."""

valid_chapters = {code for code, _ in chapter_options}
total_tokens = 0

def classify_chapter(cod_string: str) -> tuple[str, bool]:
    """Classify a CoD string into a chapter letter. Returns (prediction, success)."""
    global total_tokens
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f'Cause of death: "{cod_string}"'},
            ],
            temperature=LLM_TEMPERATURE,
            max_tokens=8,
        )
        if resp.usage:
            total_tokens += resp.usage.total_tokens
        raw = resp.choices[0].message.content.strip().upper()
        # Extract single letter
        for ch in raw:
            if ch in valid_chapters:
                return ch, True
        return raw[:1], False
    except Exception as e:
        return "?", False

# Smoke test
test_q = df_sample["query"].iloc[0]
test_gold = df_sample["gold_chapter"].iloc[0]
pred, ok = classify_chapter(test_q)
print(f"Smoke test: '{test_q}' → predicted={pred}, gold={test_gold}, match={pred==test_gold}, parsed={ok}")

In [ ]:
# Cell 5: Run classification on full sample (with checkpointing)
from tqdm.auto import tqdm

CHECKPOINT_PATH = CACHE_DIR / "checkpoint.pkl"
CHECKPOINT_EVERY = 100

# Resume from checkpoint if exists
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "rb") as f:
        results = pickle.load(f)
    print(f"Resumed from checkpoint: {len(results)} records done")
else:
    results = []  # list of (query, gold_chapter, predicted_chapter, parsed_ok)

start_idx = len(results)
queries = df_sample["query"].tolist()
golds = df_sample["gold_chapter"].tolist()

print(f"Running chapter classification: {start_idx}/{len(queries)} done, {len(queries)-start_idx} remaining")
t0 = time.time()

try:
    for i in tqdm(range(start_idx, len(queries)), initial=start_idx, total=len(queries)):
        pred, ok = classify_chapter(queries[i])
        results.append((queries[i], golds[i], pred, ok))

        if len(results) % CHECKPOINT_EVERY == 0:
            with open(CHECKPOINT_PATH, "wb") as f:
                pickle.dump(results, f)
except KeyboardInterrupt:
    print(f"\nInterrupted after {len(results)} records.")

# Final save
with open(CHECKPOINT_PATH, "wb") as f:
    pickle.dump(results, f)

elapsed = time.time() - t0
n_new = len(results) - start_idx
print(f"\nDone: {len(results)}/{len(queries)} records")
print(f"Time: {elapsed:.1f}s ({elapsed/max(n_new,1):.2f}s per record)")
print(f"Tokens used: {total_tokens:,}")

In [ ]:
# Cell 6: Evaluate
import matplotlib.pyplot as plt

df_results = pd.DataFrame(results, columns=["query", "gold", "predicted", "parsed_ok"])

df_results["correct"] = df_results["gold"] == df_results["predicted"]
accuracy = df_results["correct"].mean()
parse_rate = df_results["parsed_ok"].mean()

print(f"Records evaluated: {len(df_results)}")
print(f"Parse success rate: {parse_rate:.1%}")
print(f"Overall accuracy:   {accuracy:.1%}")
print()

# Per-chapter accuracy
ch_acc = df_results.groupby("gold")["correct"].agg(["mean", "count"]).sort_values("count", ascending=False)
ch_acc.columns = ["accuracy", "n_records"]
print("Per-chapter accuracy (sorted by frequency):")
print(ch_acc.to_string())

In [ ]:
# Cell 7: Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = sorted(df_results["gold"].unique())
cm = confusion_matrix(df_results["gold"], df_results["predicted"], labels=labels)

fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"Chapter Classification Confusion Matrix (acc={accuracy:.1%})")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Inspect misclassifications
misses = df_results[~df_results["correct"]].copy()
print(f"Misclassifications: {len(misses)}/{len(df_results)} ({len(misses)/len(df_results)*100:.1f}%)")
print()

# Most common confusion pairs
confusion_pairs = misses.groupby(["gold", "predicted"]).size().sort_values(ascending=False)
print("Top 10 confusion pairs (gold → predicted):")
print(confusion_pairs.head(10).to_string())
print()

# Show some examples
print("Sample misclassifications:")
for _, row in misses.head(15).iterrows():
    print(f"  '{row['query']:<40s}' gold={row['gold']} pred={row['predicted']}")

In [ ]:
# Cell 9: Save results
df_results.to_csv(CACHE_DIR / "chapter_results.csv", index=False)
print(f"Saved to {CACHE_DIR / 'chapter_results.csv'}")
print(f"\nSummary: {accuracy:.1%} accuracy on {len(df_results)} single-CoD records (chapter-level, 25-way)")